## Load data

In [14]:
import pandas as pd
import itertools
from scipy.stats import friedmanchisquare, wilcoxon
import statsmodels.stats.multitest as smm

# ======================
# LOAD DATA
# ======================
df = pd.read_csv("results_rev2/resdata_5iter__MCC_NTDT+TDT.csv")

# Rimuovi prima colonna se è indice
df = df.drop(columns=df.columns[0])
df

,STATIC,BINARY,DOME,LSTM,tLSTM,GRU,GRU-D,BEHRT,RETAIN,Dipole
0,0.648886,0.770833,0.745800,0.638311,0.713542,0.674791,0.799060,0.681026,0.648886,0.703646
1,0.512195,0.604524,0.563884,0.541667,0.576066,0.495209,0.674791,0.498239,0.516398,0.689222
2,0.466667,0.621585,0.576066,0.567367,0.452648,0.567367,0.712951,0.485507,0.466667,0.617670
3,0.619025,0.741941,0.800132,0.555113,0.576066,0.509749,0.620406,0.652580,0.534047,0.557364
4,0.597546,0.772249,0.802377,0.728301,0.769581,0.683749,0.888735,0.712951,0.617670,0.829621
5,0.754465,0.766032,0.707107,0.519488,0.501004,0.554164,0.638311,0.632644,0.584705,0.598958
6,0.480055,0.739098,0.681026,0.604524,0.571336,0.403390,0.764823,0.740233,0.498239,0.648181
7,0.466667,0.585570,0.576066,0.620406,0.406332,0.526810,0.630789,0.585301,0.466667,0.649292
8,0.600000,0.858955,0.829598,0.800132,0.695943,0.714901,0.804984,0.680534,0.632644,0.696627
9,0.617581,0.696627,0.737698,0.534047,0.588051,0.462910,0.649292,0.771774,0.466667,0.652580


## Friedman test

In [15]:
# ======================
# FRIEDMAN TEST
# ======================
stat, p = friedmanchisquare(*[df[col] for col in df.columns])

print("\n=== FRIEDMAN TEST ===")
print(f"Statistic: {stat:.4f}")
print(f"p-value: {p:.6f}")

if p < 0.05:
    print("➡️ Differenze globali SIGNIFICATIVE")
else:
    print("➡️ Nessuna differenza globale")

# ======================
# POST-HOC
# ======================
pairs = list(itertools.combinations(df.columns, 2))

pvals = []
results = []

for a, b in pairs:
    try:
        _, pw = wilcoxon(df[a], df[b])
    except:
        pw = 1.0
    pvals.append(pw)
    results.append((a, b, pw))

# Correzione Holm
reject, pvals_corr, _, _ = smm.multipletests(pvals, method='holm')

print("\n=== SIGNIFICANT PAIRS (Holm corrected) ===")

found = False
for i, (a, b, _) in enumerate(results):
    if reject[i]:
        found = True
        print(f"✅ {a} vs {b} → p_corr = {pvals_corr[i]:.4f}")

if not found:
    print("❌ Nessuna coppia significativa (dopo correzione)")

# ======================
# (OPZIONALE) DEBUG COMPLETO
# ======================
print("\n=== ALL PAIRS (debug) ===")
for i, (a, b, _) in enumerate(results):
    mark = "✅" if reject[i] else " "
    print(f"{mark} {a} vs {b}: raw={pvals[i]:.4f}, corr={pvals_corr[i]:.4f}")


=== FRIEDMAN TEST ===
Statistic: 117.4008
p-value: 0.000000
➡️ Differenze globali SIGNIFICATIVE

=== SIGNIFICANT PAIRS (Holm corrected) ===
✅ STATIC vs BINARY → p_corr = 0.0000
✅ STATIC vs DOME → p_corr = 0.0000
✅ STATIC vs GRU-D → p_corr = 0.0020
✅ BINARY vs LSTM → p_corr = 0.0009
✅ BINARY vs tLSTM → p_corr = 0.0000
✅ BINARY vs GRU → p_corr = 0.0002
✅ BINARY vs RETAIN → p_corr = 0.0000
✅ BINARY vs Dipole → p_corr = 0.0078
✅ DOME vs LSTM → p_corr = 0.0013
✅ DOME vs tLSTM → p_corr = 0.0000
✅ DOME vs GRU → p_corr = 0.0001
✅ DOME vs RETAIN → p_corr = 0.0000
✅ DOME vs Dipole → p_corr = 0.0228
✅ LSTM vs GRU → p_corr = 0.0083
✅ LSTM vs GRU-D → p_corr = 0.0039
✅ LSTM vs RETAIN → p_corr = 0.0014
✅ tLSTM vs GRU-D → p_corr = 0.0006
✅ GRU vs GRU-D → p_corr = 0.0000
✅ GRU vs BEHRT → p_corr = 0.0349
✅ GRU-D vs RETAIN → p_corr = 0.0000
✅ GRU-D vs Dipole → p_corr = 0.0014
✅ BEHRT vs RETAIN → p_corr = 0.0013

=== ALL PAIRS (debug) ===
✅ STATIC vs BINARY: raw=0.0000, corr=0.0000
✅ STATIC vs DOME: raw=

## Wilcoxon test

In [17]:
alpha = 0.05

# ======================
# 2) WILCOXON POST-HOC + HOLM
# ======================
pairs = list(itertools.combinations(df.columns, 2))

wilcoxon_raw_p = []
wilcoxon_results = []

for a, b in pairs:
    try:
        stat, pval = wilcoxon(df[a], df[b])
    except ValueError:
        stat, pval = None, 1.0
    wilcoxon_raw_p.append(pval)
    wilcoxon_results.append((a, b, stat, pval))

wilcoxon_reject, wilcoxon_p_corr, _, _ = smm.multipletests(
    wilcoxon_raw_p, alpha=alpha, method="holm"
)

print("\n" + "="*60)
print("WILCOXON PAIRWISE POST-HOC + HOLM")
print("="*60)

found_sig = False
for i, (a, b, stat, pval) in enumerate(wilcoxon_results):
    if wilcoxon_reject[i]:
        found_sig = True
        print(f"✅ {a} vs {b} | raw p = {pval:.6f} | Holm p = {wilcoxon_p_corr[i]:.6f}")

if not found_sig:
    print("❌ Nessuna coppia significativa dopo correzione Holm")

print("\nDettaglio completo Wilcoxon:")
for i, (a, b, stat, pval) in enumerate(wilcoxon_results):
    mark = "✅" if wilcoxon_reject[i] else "  "
    print(f"{mark} {a:>8s} vs {b:<8s} | raw p = {pval:.6f} | Holm p = {wilcoxon_p_corr[i]:.6f}")


WILCOXON PAIRWISE POST-HOC + HOLM
✅ STATIC vs BINARY | raw p = 0.000000 | Holm p = 0.000003
✅ STATIC vs DOME | raw p = 0.000000 | Holm p = 0.000007
✅ STATIC vs GRU-D | raw p = 0.000068 | Holm p = 0.001962
✅ BINARY vs LSTM | raw p = 0.000027 | Holm p = 0.000906
✅ BINARY vs tLSTM | raw p = 0.000000 | Holm p = 0.000012
✅ BINARY vs GRU | raw p = 0.000005 | Holm p = 0.000189
✅ BINARY vs RETAIN | raw p = 0.000000 | Holm p = 0.000003
✅ BINARY vs Dipole | raw p = 0.000287 | Holm p = 0.007751
✅ DOME vs LSTM | raw p = 0.000038 | Holm p = 0.001259
✅ DOME vs tLSTM | raw p = 0.000000 | Holm p = 0.000007
✅ DOME vs GRU | raw p = 0.000003 | Holm p = 0.000095
✅ DOME vs RETAIN | raw p = 0.000000 | Holm p = 0.000003
✅ DOME vs Dipole | raw p = 0.000912 | Holm p = 0.022797
✅ LSTM vs GRU | raw p = 0.000318 | Holm p = 0.008274
✅ LSTM vs GRU-D | raw p = 0.000140 | Holm p = 0.003917
✅ LSTM vs RETAIN | raw p = 0.000045 | Holm p = 0.001408
✅ tLSTM vs GRU-D | raw p = 0.000018 | Holm p = 0.000638
✅ GRU vs GRU-D |

In [21]:
import numpy as np

methods = list(df.columns)
n = len(methods)

# matrice p-value corretti
p_matrix = np.ones((n, n))

# riempi matrice
k = 0
for i in range(n):
    for j in range(i+1, n):
        p_matrix[i, j] = wilcoxon_p_corr[k]
        p_matrix[j, i] = wilcoxon_p_corr[k]
        k += 1

# ======================
# STAMPA LATEX
# ======================
print("\n=== LATEX MATRIX (Holm corrected p-values) ===\n")

print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Pairwise Wilcoxon test (Holm-corrected p-values).}")
print("\\begin{tabular}{l" + "c"*n + "}")
print("\\hline")

# header
header = " & " + " & ".join(methods) + " \\\\"
print(header)
print("\\hline")

# rows
for i in range(n):
    row = [methods[i]]
    for j in range(n):
        if i == j:
            row.append("--")
        else:
            val = p_matrix[i, j]
            if val < 0.001:
                txt = "$<0.001$"
            else:
                txt = f"{val:.3f}"

            # bold se significativo
            if val < alpha:
                txt = "\\textbf{" + txt + "}"

            row.append(txt)
    print(" & ".join(row) + " \\\\")

print("\\hline")
print("\\end{tabular}")
print("\\end{table}")


=== LATEX MATRIX (Holm corrected p-values) ===

\begin{table}[h]
\centering
\caption{Pairwise Wilcoxon test (Holm-corrected p-values).}
\begin{tabular}{lcccccccccc}
\hline
 & STATIC & BINARY & DOME & LSTM & tLSTM & GRU & GRU-D & BEHRT & RETAIN & Dipole \\
\hline
STATIC & -- & \textbf{$<0.001$} & \textbf{$<0.001$} & 0.206 & 1.000 & 1.000 & \textbf{0.002} & 0.052 & 0.425 & 1.000 \\
BINARY & \textbf{$<0.001$} & -- & 1.000 & \textbf{$<0.001$} & \textbf{$<0.001$} & \textbf{$<0.001$} & 1.000 & 0.062 & \textbf{$<0.001$} & \textbf{0.008} \\
DOME & \textbf{$<0.001$} & 1.000 & -- & \textbf{0.001} & \textbf{$<0.001$} & \textbf{$<0.001$} & 1.000 & 0.101 & \textbf{$<0.001$} & \textbf{0.023} \\
LSTM & 0.206 & \textbf{$<0.001$} & \textbf{0.001} & -- & 0.319 & \textbf{0.008} & \textbf{0.004} & 1.000 & \textbf{0.001} & 1.000 \\
tLSTM & 1.000 & \textbf{$<0.001$} & \textbf{$<0.001$} & 0.319 & -- & 1.000 & \textbf{$<0.001$} & 0.324 & 1.000 & 1.000 \\
GRU & 1.000 & \textbf{$<0.001$} & \textbf{$<0.001$} & 

### Rank con Wilcoxon (Win cout)

In [28]:
import pandas as pd

methods = list(df.columns)
n = len(methods)

# inizializza punteggi
wins = {m: 0 for m in methods}
losses = {m: 0 for m in methods}

k = 0
for i in range(n):
    for j in range(i+1, n):
        a = methods[i]
        b = methods[j]
        
        if wilcoxon_reject[k]:  # significativo
            # chi ha media maggiore?
            if df[a].mean() > df[b].mean():
                wins[a] += 1
                losses[b] += 1
            else:
                wins[b] += 1
                losses[a] += 1
        k += 1

# crea dataframe ranking
ranking_df = pd.DataFrame({
    "wins": wins,
    "losses": losses
})

ranking_df["score"] = ranking_df["wins"] - ranking_df["losses"]

ranking_df = ranking_df.sort_values(by="score", ascending=False)

print("\n=== WILCOXON RANKING (win-based) ===")
print(ranking_df.to_latex())


=== WILCOXON RANKING (win-based) ===
\begin{tabular}{lrrr}
\toprule
 & wins & losses & score \\
\midrule
BINARY & 6 & 0 & 6 \\
DOME & 6 & 0 & 6 \\
GRU-D & 6 & 0 & 6 \\
BEHRT & 2 & 0 & 2 \\
LSTM & 2 & 3 & -1 \\
STATIC & 0 & 3 & -3 \\
Dipole & 0 & 3 & -3 \\
tLSTM & 0 & 3 & -3 \\
GRU & 0 & 5 & -5 \\
RETAIN & 0 & 5 & -5 \\
\bottomrule
\end{tabular}



## ANOVA

In [ ]:
# ======================
# 3) REPEATED MEASURES ANOVA
# ======================
# Serve il formato long: subject, method, score
df_long = df.copy()
df_long["subject"] = range(len(df_long))
df_long = df_long.melt(id_vars="subject", var_name="method", value_name="score")

anova = AnovaRM(df_long, depvar="score", subject="subject", within=["method"])
anova_res = anova.fit()

print("\n" + "="*60)
print("REPEATED MEASURES ANOVA")
print("="*60)
print(anova_res)

## Paired T-Test + Holm

In [ ]:
# ======================
# 4) PAIRED T-TESTS + HOLM
# ======================
ttest_raw_p = []
ttest_results = []

for a, b in pairs:
    stat, pval = ttest_rel(df[a], df[b])
    ttest_raw_p.append(pval)
    ttest_results.append((a, b, stat, pval))

ttest_reject, ttest_p_corr, _, _ = smm.multipletests(
    ttest_raw_p, alpha=alpha, method="holm"
)

print("\n" + "="*60)
print("PAIRED T-TESTS + HOLM")
print("="*60)

found_sig = False
for i, (a, b, stat, pval) in enumerate(ttest_results):
    if ttest_reject[i]:
        found_sig = True
        print(f"✅ {a} vs {b} | t = {stat:.6f} | raw p = {pval:.6f} | Holm p = {ttest_p_corr[i]:.6f}")

if not found_sig:
    print("❌ Nessuna coppia significativa dopo correzione Holm")

print("\nDettaglio completo t-test:")
for i, (a, b, stat, pval) in enumerate(ttest_results):
    mark = "✅" if ttest_reject[i] else "  "
    print(f"{mark} {a:>8s} vs {b:<8s} | t = {stat:>10.6f} | raw p = {pval:.6f} | Holm p = {ttest_p_corr[i]:.6f}")


WILCOXON PAIRWISE POST-HOC + HOLM
✅ STATIC vs BINARY | raw p = 0.000000 | Holm p = 0.000003
✅ STATIC vs DOME | raw p = 0.000000 | Holm p = 0.000007
✅ STATIC vs GRU-D | raw p = 0.000068 | Holm p = 0.001962
✅ BINARY vs LSTM | raw p = 0.000027 | Holm p = 0.000906
✅ BINARY vs tLSTM | raw p = 0.000000 | Holm p = 0.000012
✅ BINARY vs GRU | raw p = 0.000005 | Holm p = 0.000189
✅ BINARY vs RETAIN | raw p = 0.000000 | Holm p = 0.000003
✅ BINARY vs Dipole | raw p = 0.000287 | Holm p = 0.007751
✅ DOME vs LSTM | raw p = 0.000038 | Holm p = 0.001259
✅ DOME vs tLSTM | raw p = 0.000000 | Holm p = 0.000007
✅ DOME vs GRU | raw p = 0.000003 | Holm p = 0.000095
✅ DOME vs RETAIN | raw p = 0.000000 | Holm p = 0.000003
✅ DOME vs Dipole | raw p = 0.000912 | Holm p = 0.022797
✅ LSTM vs GRU | raw p = 0.000318 | Holm p = 0.008274
✅ LSTM vs GRU-D | raw p = 0.000140 | Holm p = 0.003917
✅ LSTM vs RETAIN | raw p = 0.000045 | Holm p = 0.001408
✅ tLSTM vs GRU-D | raw p = 0.000018 | Holm p = 0.000638
✅ GRU vs GRU-D |

## Medie e ranking

In [ ]:
# ======================
# 5) MEDIE E RANKING
# ======================
print("\n" + "="*60)
print("MEAN SCORES")
print("="*60)
print(df.mean().sort_values(ascending=False))

print("\n" + "="*60)
print("AVERAGE RANKS (1 = migliore)")
print("="*60)
ranks = df.rank(axis=1, ascending=False, method="average")
print(ranks.mean().sort_values())